In [1]:
!pip install -U transformers peft scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 72.6 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 90.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 646.8/646.8 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 93.1 MB/s eta 0:00:00:00:01
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: transformers
    Found existing install

In [3]:
!pip uninstall -y torchao
!pip install torchao>=0.16.0
!pip install -U peft transformers

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [4]:
# ------------------------------------------------------------
# 1) Imports
# ------------------------------------------------------------
from transformers import AutoTokenizer, AutoModel, TrainingArguments, Trainer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from peft import LoraConfig, get_peft_model
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

# ------------------------------------------------------------
# 2) Load Data
# ------------------------------------------------------------
df = pd.read_csv("/kaggle/input/datasets/shuvo128/loraaa/IMDB_Cleaned (1).csv")

text_column = "cleaned_review" if "cleaned_review" in df.columns else "review"
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

texts  = df[text_column].astype(str).tolist()
labels = (df["sentiment"] == "positive").astype(int).tolist()

# ------------------------------------------------------------
# 3) Split (40k / 5k / 5k)
# ------------------------------------------------------------
train_texts = texts[:40000]
train_labels = labels[:40000]

val_texts = texts[40000:45000]
val_labels = labels[40000:45000]

test_texts = texts[45000:50000]
test_labels = labels[45000:50000]

# ------------------------------------------------------------
# 4) TF-IDF
# ------------------------------------------------------------
tfidf = TfidfVectorizer(max_features=8000, ngram_range=(1,2))
tfidf.fit(train_texts)

train_tfidf = tfidf.transform(train_texts).toarray()
val_tfidf   = tfidf.transform(val_texts).toarray()
test_tfidf  = tfidf.transform(test_texts).toarray()

# ------------------------------------------------------------
# 5) Tokenizer
# ------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained("google/electra-base-discriminator")

def tokenize(texts):
    return tokenizer(texts, truncation=True, padding=True, max_length=256)

train_enc = tokenize(train_texts)
val_enc   = tokenize(val_texts)
test_enc  = tokenize(test_texts)

# ------------------------------------------------------------
# 6) Dataset
# ------------------------------------------------------------
class HybridDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, tfidf, labels):
        self.encodings = encodings
        self.tfidf = tfidf
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["tfidf"] = torch.tensor(self.tfidf[idx], dtype=torch.float)
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = HybridDataset(train_enc, train_tfidf, train_labels)
val_dataset   = HybridDataset(val_enc, val_tfidf, val_labels)
test_dataset  = HybridDataset(test_enc, test_tfidf, test_labels)

# ------------------------------------------------------------
# 7) LoRA + ELECTRA Model
# ------------------------------------------------------------
class ElectraLoRAHybrid(nn.Module):
    def __init__(self, model_name, tfidf_dim, num_labels):
        super().__init__()

        base_model = AutoModel.from_pretrained(model_name)

        # LoRA config
        lora_config = LoraConfig(
            r=8,
            lora_alpha=16,
            target_modules=["query", "key"],  # safe option
            lora_dropout=0.1,
            bias="none"
        )

        self.electra = get_peft_model(base_model, lora_config)

        hidden_size = self.electra.config.hidden_size

        # TF-IDF projection
        self.tfidf_proj = nn.Linear(tfidf_dim, hidden_size)

        # Fusion (simple + effective)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_labels)
        )

    def forward(self, input_ids, attention_mask, tfidf, labels=None):
        outputs = self.electra(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_output = outputs.last_hidden_state[:, 0, :]

        tfidf_embed = self.tfidf_proj(tfidf)

        combined = torch.cat([cls_output, tfidf_embed], dim=1)

        logits = self.classifier(combined)

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss(label_smoothing=0.1)(logits, labels)

        return {"loss": loss, "logits": logits}

# Initialize model
model = ElectraLoRAHybrid(
    model_name="google/electra-base-discriminator",
    tfidf_dim=8000,
    num_labels=2
)

# Show trainable params
model.electra.print_trainable_parameters()

# ------------------------------------------------------------
# 8) Metrics
# ------------------------------------------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary'
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

# ------------------------------------------------------------
# 9) Training
# ------------------------------------------------------------
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_steps=100,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# ------------------------------------------------------------
# 10) Train
# ------------------------------------------------------------
trainer.train()

# ------------------------------------------------------------
# 11) Evaluate
# ------------------------------------------------------------
print("\nValidation:")
print(trainer.evaluate())

print("\nTest:")
test_results = trainer.predict(test_dataset)

test_preds = np.argmax(test_results.predictions, axis=1)

precision, recall, f1, _ = precision_recall_fscore_support(
    test_labels, test_preds, average='binary'
)
acc = accuracy_score(test_labels, test_preds)

print({
    "accuracy": acc,
    "precision": precision,
    "recall": recall,
    "f1": f1
})

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraModel LOAD REPORT from: google/electra-base-discriminator
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
electra.embeddings_project.weight                 | UNEXPECTED |  | 
electra.embeddings_project.bias                   | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
discriminator_predictions.dense.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


trainable params: 294,912 || all params: 109,186,560 || trainable%: 0.2701


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
100,0.693054
200,0.685920
300,0.681380
400,0.668828
500,0.657922
600,0.631420
700,0.585091
800,0.497127
900,0.434678
1000,0.394925



Validation:


Training Loss,Validation Loss,Step,Accuracy,Precision,Recall,F1
0.303940,0.315032,10000,0.932000,0.923168,0.941058,0.932027


{'eval_loss': 0.31503188610076904, 'eval_accuracy': 0.932, 'eval_precision': 0.9231683168316832, 'eval_recall': 0.9410577311263625, 'eval_f1': 0.9320271891243502}

Test:


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'accuracy': 0.9406, 'precision': 0.9363707776904949, 'recall': 0.9464073044859072, 'f1': 0.9413622902270484}
